In [0]:
%run "../../utils"

In [0]:
# Define the target Delta table name
target = f"{silver_schema}.epa_vehicles"

In [0]:
df_bronze_veh = spark.table(f"{bronze_schema}.epa_vehicles_raw")

In [0]:
# Transform bronze vehicle data to silver layer with selected and cleaned columns
df_silver_veh = (
    df_bronze_veh
      # Cast 'id' to BIGINT for vehicle_id
      .withColumn("vehicle_id", expr("try_cast(id as BIGINT)"))
      # Filter out rows where vehicle_id is null
      .filter(col("vehicle_id").isNotNull())
      # Select and rename relevant columns, cast types as needed
      .select(
          col("vehicle_id"),
          col("year").cast("int"),
          col("make"),
          col("model"),
          col("fueltype").alias("fuel_type"),
          col("city08").cast("double").alias("city_mpg"),
          col("highway08").cast("double").alias("highway_mpg"),
          col("comb08").cast("double").alias("combined_mpg"),
          current_timestamp().alias("ingestion_ts")  # Add ingestion timestamp
      )
      # Remove duplicate vehicle_id entries
      .dropDuplicates(["vehicle_id"])
)

In [0]:
# Check if the target table exists
if spark.catalog.tableExists(target):
    # If the table exists, perform an upsert (merge) operation
    dt = DeltaTable.forName(spark, target)
    (dt.alias("t")
      .merge(df_silver_veh.alias("s"), "t.vehicle_id = s.vehicle_id")  # Match on vehicle_id
      .whenMatchedUpdateAll()  # Update all columns if matched
      .whenNotMatchedInsertAll()  # Insert all columns if not matched
      .execute())
else:
    # If the table does not exist, create it with the new data
    df_silver_veh.write.format("delta").mode("overwrite").saveAsTable(target)